Part B: Setup and data (Colab cells)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SenCLIP
!pip install -q pytorch-lightning rasterio opencv-python scikit-learn ftfy regex huggingface_hub
!pip install -q git+https://github.com/openai/CLIP.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SenCLIP
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np, torch, random, gc
from huggingface_hub import hf_hub_download

emb = hf_hub_download("pallavijainpj/SenCLIP",
      "LUCAS_CLIP_Frozen_Embeddings/lucas_clip_emb1024_RN50.pt", local_dir="/content/emb")
!wget -q -O /content/paths.txt "https://zenodo.org/records/14231252/files/existing_sentinel_paths.txt?download=1"

paths = [l.strip().lstrip('./') for l in open('/content/paths.txt') if l.strip()]
print(len(paths), paths[:3])

d = torch.load(emb)
k0 = next(iter(d)); print(len(d), k0, d[k0].shape, d[k0].dtype)

def pid(p): return f'{p.split("/")[0]}_{p.split("/")[-1].split(".")[0].split("_")[0]}'
ok = [p for p in paths if all(f'{pid(p)}{s}' in d for s in 'WENS')]
print("paths with all 4 embeddings:", len(ok))

random.seed(0)
sub = random.sample(ok, 8000)
np.save('/content/small_paths.npy', np.array(sub))
del d; gc.collect()

235210 ['EE/51864104.tif', 'EE/52624068.tif', 'EE/51704092.tif']
948632 PT_26381954W torch.Size([1, 1024]) torch.float32
paths with all 4 embeddings: 235210


8

In [ ]:
import subprocess, os, glob, tarfile, zlib, time
import numpy as np

URL = "https://zenodo.org/records/14231252/files/Sen2LUCAS.tar.gz?download=1"
N_PARTS, PART = 8, 150_000_000          # 8 x 150 MB = 1.2 GB
t0 = time.time()

# 1) download 8 pieces in parallel
procs = []
for i in range(N_PARTS):
    s, e = i * PART, (i + 1) * PART - 1
    procs.append(subprocess.Popen(
        ["curl", "-s", "-L", "-g", "--retry", "5", "-r", f"{s}-{e}", "-o", f"/content/p{i}", URL]))
for p in procs:
    p.wait()

# 2) join the pieces
with open("/content/part.tar.gz", "wb") as out:
    for i in range(N_PARTS):
        with open(f"/content/p{i}", "rb") as f:
            out.write(f.read())
        os.remove(f"/content/p{i}")
print(f"downloaded {os.path.getsize('/content/part.tar.gz')/1e6:.0f} MB in {time.time()-t0:.0f}s")

# 3) extract the patches that have embeddings
subprocess.run(["wget", "-q", "-O", "/content/paths.txt",
                "https://zenodo.org/records/14231252/files/existing_sentinel_paths.txt?download=1"])
want = set(l.strip().lstrip('./') for l in open('/content/paths.txt') if l.strip())
os.makedirs('/content/Sen2LUCAS', exist_ok=True)
found, names = 0, []
try:
    with tarfile.open('/content/part.tar.gz', 'r|gz') as tar:
        for m in tar:
            if not m.isfile():
                continue
            n = m.name.lstrip('./')
            if len(names) < 3:
                names.append(m.name)
            for cand in (n, '/'.join(n.split('/')[1:])):
                if cand in want:
                    m.name = cand
                    tar.extract(m, '/content/Sen2LUCAS')
                    found += 1
                    break
            if found >= 8000:
                break
except (EOFError, tarfile.ReadError, zlib.error, OSError) as e:
    print("reached end of partial file (normal):", type(e).__name__)

# 4) save the list of extracted patches
got = [p.replace('/content/Sen2LUCAS/', '') for p in glob.glob('/content/Sen2LUCAS/**/*.tif', recursive=True)]
np.save('/content/small_paths.npy', np.array(got))
print("extracted:", len(got), "| first names:", names)

downloaded 1200 MB in 250s
reached end of partial file (normal): ReadError
extracted: 0 | first names: ['home/pallavi/DATA/Datasets/Sentinel2_Lucas/sentinel_paths.txt', 'home/pallavi/DATA/Datasets/Sentinel2_Lucas/LU/40602970.tif', 'home/pallavi/DATA/Datasets/Sentinel2_Lucas/LU/40282972.tif']


In [ ]:
import tarfile, zlib, os, glob
import numpy as np

want = set(l.strip().lstrip('./') for l in open('/content/paths.txt') if l.strip())
KEY = 'Sentinel2_Lucas/'
os.makedirs('/content/Sen2LUCAS', exist_ok=True)
found = 0
try:
    with tarfile.open('/content/part.tar.gz', 'r|gz') as tar:
        for m in tar:
            if not m.isfile():
                continue
            i = m.name.find(KEY)
            if i < 0:
                continue
            rel = m.name[i + len(KEY):]
            if rel in want:
                m.name = rel
                tar.extract(m, '/content/Sen2LUCAS')
                found += 1
                if found % 1000 == 0:
                    print(found)
                if found >= 8000:
                    break
except (EOFError, tarfile.ReadError, zlib.error, OSError) as e:
    print("reached end of partial file (normal):", type(e).__name__)

got = [p.replace('/content/Sen2LUCAS/', '') for p in glob.glob('/content/Sen2LUCAS/**/*.tif', recursive=True)]
np.save('/content/small_paths.npy', np.array(got))
print("extracted:", len(got))

/tmp/ipykernel_15955/829373741.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(m, '/content/Sen2LUCAS')


1000
2000
3000
4000
5000
6000
7000
8000
extracted: 8000


In [ ]:
emb = "/content/emb/LUCAS_CLIP_Frozen_Embeddings/lucas_clip_emb1024_RN50.pt"
import sys; sys.path.insert(0, '/content/drive/MyDrive/SenCLIP')
from dataset_sentinel import SenDataset
ds = SenDataset('/content/Sen2LUCAS/', emb, '/content/small_paths.npy')
a, b = ds[0]; print(a.shape, b.shape)

8000 sentinel images found
torch.Size([4, 1024]) torch.Size([3, 224, 224])


/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [ ]:
import torch, numpy as np, gc
emb = "/content/emb/LUCAS_CLIP_Frozen_Embeddings/lucas_clip_emb1024_RN50.pt"
paths = np.load('/content/small_paths.npy')

def pid(p): return f'{p.split("/")[0]}_{p.split("/")[-1].split(".")[0].split("_")[0]}'
need = {f'{pid(p)}{s}' for p in paths for s in 'WENS'}

d = torch.load(emb)
small = {k: d[k].clone() for k in need}
del d; gc.collect()
torch.save(small, '/content/emb_small.pt')
print("saved keys:", len(small))

saved keys: 32000


Part C: Training

In [ ]:
%cd /content/drive/MyDrive/SenCLIP
!python main.py --root_data_dir /content/Sen2LUCAS/ \
 --emb_path /content/emb_small.pt \
 --data_path_list /content/small_paths.npy \
 --saved_model /content/drive/MyDrive/SenCLIP/ckpt --version_fold rn50_avg_test \
 --ARCH RN50 --pooling avgpool --NUM_EPOCHS 1 --device 0 \
 --LR 1e-5 --BATCH_SIZE 32 --NUM_WORKERS 0

/content/drive/MyDrive/SenCLIP
8000 sentinel images found
/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
Sample input shapes: torch.Size([4, 1024]) torch.Size([3, 224, 224])
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
Using Adaptive Average Pool.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━

In [ ]:
%cd /content/drive/MyDrive/SenCLIP
!python main.py --root_data_dir /content/Sen2LUCAS/ \
 --emb_path /content/emb_small.pt \
 --data_path_list /content/small_paths.npy \
 --saved_model /content/drive/MyDrive/SenCLIP/ckpt --version_fold rn50_avg_run1 \
 --ARCH RN50 --pooling avgpool --NUM_EPOCHS 20 --device 0 \
 --LR 1e-5 --BATCH_SIZE 32 --NUM_WORKERS 0

/content/drive/MyDrive/SenCLIP
8000 sentinel images found
/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
Sample input shapes: torch.Size([4, 1024]) torch.Size([3, 224, 224])
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
Using Adaptive Average Pool.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━

Part D: Convert the checkpoint

In [ ]:
import glob, pandas as pd
f = sorted(glob.glob('/content/drive/MyDrive/SenCLIP/logs/lightning_logs/version_*/metrics.csv'))[-1]
df = pd.read_csv(f)
print(df[['epoch','train_loss_epoch']].dropna().to_string())

     epoch  train_loss_epoch
26     0.0          6.047126
53     1.0          5.958408
80     2.0          5.890243
107    3.0          5.803535
134    4.0          5.729921
161    5.0          5.686966
188    6.0          5.615129
215    7.0          5.589251
242    8.0          5.541656
269    9.0          5.500678
296   10.0          5.459195
323   11.0          5.414707
350   12.0          5.369208
377   13.0          5.337593
404   14.0          5.290640
431   15.0          5.272818
458   16.0          5.226690
485   17.0          5.207479
512   18.0          5.150601
539   19.0          5.104902


In [ ]:
import torch
sd = torch.load("/content/drive/MyDrive/SenCLIP/ckpt/rn50_avg_run1/last.ckpt", map_location="cpu")["state_dict"]
p = "learner.encoder.model."
torch.save({k[len(p):]: v for k, v in sd.items() if k.startswith(p)},
           "/content/drive/MyDrive/SenCLIP/SenCLIP_mine.ckpt")
print("saved")

saved


Part E: Evaluate on EuroSAT

In [ ]:
import torchvision
torchvision.datasets.EuroSAT(root="/content/data", download=True)
!mkdir -p /content/Datasets/EuroSAT
!ln -sfn /content/data/eurosat/2750 /content/Datasets/EuroSAT/2750
!ls /content/Datasets/EuroSAT/2750

100%|██████████| 94.3M/94.3M [00:01<00:00, 56.1MB/s]


AnnualCrop  HerbaceousVegetation  Industrial  PermanentCrop  River
Forest	    Highway		  Pasture     Residential    SeaLake


In [ ]:
%cd /content/drive/MyDrive/SenCLIP
print("=== AERIAL prompts ===")
!python zeroshot.py --model_arch RN50 --ckpt_path SenCLIP_mine.ckpt --root_dir /content/ --template_path prompt_eurosat_aerial_mixed.json 2>&1 | grep "Top-1"
print("=== GROUND prompts ===")
!python zeroshot.py --model_arch RN50 --ckpt_path SenCLIP_mine.ckpt --root_dir /content/ --template_path prompt_eurosat_ground_mixed.json 2>&1 | grep "Top-1"
print("=== GROUND+AERIAL prompts ===")
!python zeroshot.py --model_arch RN50 --ckpt_path SenCLIP_mine.ckpt --root_dir /content/ --template_path prompt_eurosat_ground_aerial_mixed.json 2>&1 | grep "Top-1"

/content/drive/MyDrive/SenCLIP
=== AERIAL prompts ===
=== GROUND prompts ===
=== GROUND+AERIAL prompts ===


In [ ]:
import clip, torch
m, _ = clip.load("RN50", device="cpu")
torch.save(m.state_dict(), "/content/drive/MyDrive/SenCLIP/CLIP_orig.ckpt")

!sed -e 's/(0.347, 0.376, 0.296)/(0.48145466, 0.4578275, 0.40821073)/' -e 's/(0.269, 0.261, 0.276)/(0.26862954, 0.26130258, 0.27577711)/' zeroshot.py > zeroshot_clip.py

print("=== CLIP AERIAL ===")
!python zeroshot_clip.py --model_arch RN50 --ckpt_path CLIP_orig.ckpt --root_dir /content/ --template_path prompt_eurosat_aerial_mixed.json 2>&1 | grep "Top-1"
print("=== CLIP GROUND ===")
!python zeroshot_clip.py --model_arch RN50 --ckpt_path CLIP_orig.ckpt --root_dir /content/ --template_path prompt_eurosat_ground_mixed.json 2>&1 | grep "Top-1"
print("=== CLIP GROUND+AERIAL ===")
!python zeroshot_clip.py --model_arch RN50 --ckpt_path CLIP_orig.ckpt --root_dir /content/ --template_path prompt_eurosat_ground_aerial_mixed.json 2>&1 | grep "Top-1"

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


=== CLIP AERIAL ===
=== CLIP GROUND ===
=== CLIP GROUND+AERIAL ===
